# Books & Interactions Processing Pipeline: Young Adult

Final curation pipeline for the Goodreads Young Adult category. The cleaning and feature engineering choices are grounded in `EDA_YoungAdult.ipynb`: role-filtered authorship, YA shelf taxonomy, separated demand vs satisfaction signals, cold-start routing, user rating bias, valid publication years, and transformed long-tail counts.

## Section 0 — Setup

Use the shared category configuration so raw and processed paths stay aligned with the rest of the project. Both books and interactions are processed in chunks to avoid holding large Goodreads files in memory.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import gc
import json
import os
from pathlib import Path
import sys
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import BOOK_NUMERIC_COLUMNS, CATEGORIES, GOODREADS_DATE_COLUMNS
from src.utils.cleaning import empty_strings_to_na, normalize_review_text, parse_bool_series
from src.utils.io import read_jsonl_chunks, safe_write_parquet

CATEGORY = 'young_adult'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOOKS_CURATED_PATH = OUTPUT_DIR / 'books_curated.parquet'
INTERACTIONS_CURATED_PATH = OUTPUT_DIR / 'interactions_curated.parquet'
USER_FEATURES_PATH = OUTPUT_DIR / 'user_features.parquet'
BOOK_INTERACTION_FEATURES_PATH = OUTPUT_DIR / 'book_interaction_features.parquet'
SUMMARY_PATH = OUTPUT_DIR / 'curation_summary.json'

BOOK_CHUNKSIZE = 50_000
INTERACTION_CHUNKSIZE = 200_000
COLD_START_THRESHOLD = 10
PUBLICATION_YEAR_MIN = 1450
PUBLICATION_YEAR_MAX = 2026
THEME_COUNT_COL = 'genre_theme_count'
VALID_ENGAGEMENT_MODES = {'shelf_only', 'rating_only', 'review', 'read_no_rating'}
PUBLICATION_PERIOD_BINS = [1450, 1800, 1900, 1950, 1980, 2000, 2010, 2027]
PUBLICATION_PERIOD_LABELS = ['1450-1799', '1800-1899', '1900-1949', '1950-1979', '1980-1999', '2000-2009', '2010-2026']

print(cfg.display_name)
print(cfg.books_file)
print(cfg.interactions_file)
print(OUTPUT_DIR)

Young Adult
/home/nakato/projects/BigBook/data/raw/goodreads_books_young_adult.json.gz
/home/nakato/projects/BigBook/data/raw/goodreads_interactions_young_adult.json.gz
/home/nakato/projects/BigBook/data/processed/young_adult


## Section 1 — Helper Functions

Helpers are local to this notebook to keep `src/utils` unchanged. They mirror the EDA decisions and keep the full pipeline chunk-safe.

In [10]:
THEME_GROUPS = {'ya_fantasy': ['ya-fantasy', 'fantasy-ya', 'fantasy'], 'ya_dystopia': ['dystopia', 'dystopian', 'post-apocalyptic'], 'ya_contemporary': ['ya-contemporary', 'contemporary', 'realistic-fiction'], 'ya_romance': ['ya-romance', 'teen-romance', 'romance'], 'ya_paranormal': ['paranormal', 'supernatural', 'vampires', 'werewolves'], 'ya_scifi': ['science-fiction', 'sci-fi', 'science-fiction-ya'], 'middle_grade': ['middle-grade', 'childrens', 'juvenile', 'kids'], 'coming_of_age': ['coming-of-age', 'bildungsroman', 'teen', 'teens']}


def parse_goodreads_dates_inplace(df: pd.DataFrame) -> pd.DataFrame:
    for col in GOODREADS_DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    return df


def safe_to_numeric_inplace(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def parse_bool_column(series: pd.Series) -> pd.Series:
    return parse_bool_series(series) if len(series) else pd.Series(dtype='boolean')


def json_dumps_or_na(value: Any) -> str | pd.NA:
    if value is None:
        return pd.NA
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return pd.NA
    return json.dumps(value, ensure_ascii=False)


def clean_string_column(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().replace('', pd.NA)


def normalize_review_text_series(series: pd.Series) -> pd.Series:
    out = pd.Series(pd.NA, index=series.index, dtype='object')
    mask = series.notna()
    if mask.any():
        out.loc[mask] = series.loc[mask].map(normalize_review_text)
    return out


def features_from_nested(series: pd.Series, extractor) -> pd.DataFrame:
    return pd.DataFrame(series.map(extractor).tolist(), index=series.index)


def has_theme(names: list[str], keywords: list[str]) -> bool:
    return any(any(keyword in name for keyword in keywords) for name in names)


def extract_author_features(authors: Any) -> dict[str, Any]:
    if not isinstance(authors, list):
        return {
            'primary_author_id_role_filtered': pd.NA,
            'author_fallback_id': pd.NA,
            'author_count': 0,
            'non_primary_role_count': 0,
            'primary_author_role': pd.NA,
        }
    valid = [item for item in authors if isinstance(item, dict) and item.get('author_id')]
    role_filtered = [item for item in valid if str(item.get('role', '')) == '']
    fallback = valid[0] if valid else None
    primary = role_filtered[0] if role_filtered else None
    return {
        'primary_author_id_role_filtered': str(primary.get('author_id')) if primary else pd.NA,
        'author_fallback_id': str(fallback.get('author_id')) if fallback else pd.NA,
        'author_count': len(valid),
        'non_primary_role_count': sum(1 for item in valid if str(item.get('role', '')) != ''),
        'primary_author_role': str(primary.get('role', '')) if primary else pd.NA,
    }


def extract_series_features(series_value: Any) -> dict[str, Any]:
    count = len(series_value) if isinstance(series_value, list) else 0
    return {
        'series_count': count,
        'is_in_series': count > 0,
        'series_json': json_dumps_or_na(series_value) if count else pd.NA,
    }


def extract_similar_books_features(value: Any) -> dict[str, Any]:
    count = len(value) if isinstance(value, list) else 0
    return {
        'similar_books_count': count,
        'similar_books_json': json_dumps_or_na(value) if count else pd.NA,
    }


def extract_shelf_features(shelves: Any, top_n: int = 30) -> dict[str, Any]:
    if not isinstance(shelves, list):
        base = {'to_read_count': np.nan, 'shelf_count': 0, 'top_shelves': pd.NA, 'top_shelves_json': pd.NA}
        for theme in THEME_GROUPS:
            base[f'theme_{theme}'] = False
        base[THEME_COUNT_COL] = 0
        return base

    cleaned = []
    to_read_count = np.nan
    for item in shelves:
        if not isinstance(item, dict):
            continue
        name = item.get('name')
        if not name:
            continue
        count = pd.to_numeric(item.get('count'), errors='coerce')
        name = str(name)
        if name.lower() == 'to-read':
            to_read_count = count
        cleaned.append({'name': name, 'count': None if pd.isna(count) else int(count)})

    top = cleaned[:top_n]
    names = [item['name'].lower() for item in cleaned]
    base = {
        'to_read_count': to_read_count,
        'shelf_count': len(cleaned),
        'top_shelves': '|'.join(item['name'] for item in top) if top else pd.NA,
        'top_shelves_json': json.dumps(top, ensure_ascii=False) if top else pd.NA,
    }
    theme_count = 0
    for theme, keywords in THEME_GROUPS.items():
        value = has_theme(names, keywords)
        base[f'theme_{theme}'] = value
        theme_count += int(value)
    base[THEME_COUNT_COL] = theme_count
    return base


def add_log_and_cap_features(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors='coerce')
        df[f'{col}_log1p'] = np.log1p(values.clip(lower=0))
        cap = float(values.quantile(0.99))
        df[f'{col}_p99_capped'] = values.clip(upper=cap).astype('float64')
    return df


def confidence_bucket(count: pd.Series) -> pd.Series:
    return pd.cut(count.fillna(0), bins=[-1, 0, 9, 49, np.inf], labels=['none', 'low', 'medium', 'high']).astype('string')


def cleanup_output_files() -> None:
    for path in [BOOKS_CURATED_PATH, INTERACTIONS_CURATED_PATH, USER_FEATURES_PATH, BOOK_INTERACTION_FEATURES_PATH, SUMMARY_PATH]:
        if path.exists():
            path.unlink()


def write_parquet_chunk(path: Path, chunk: pd.DataFrame, writer: pq.ParquetWriter | None) -> pq.ParquetWriter:
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(path, table.schema, compression='snappy')
    writer.write_table(table)
    return writer

## Section 2 — Interaction First Pass: Aggregates

The full interactions file is processed in chunks. This first pass computes user features, book interaction features, engagement distributions, and validation counters without materializing the final parquet yet.

In [3]:
def process_interaction_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = empty_strings_to_na(chunk.copy())
    parse_goodreads_dates_inplace(chunk)

    chunk['book_id'] = chunk['book_id'].astype('string')
    chunk['user_id'] = chunk['user_id'].astype('string')
    if 'review_id' in chunk.columns:
        chunk['review_id'] = chunk['review_id'].astype('string')

    rating = pd.to_numeric(chunk['rating'], errors='coerce') if 'rating' in chunk.columns else pd.Series(np.nan, index=chunk.index)
    chunk['rating_clean'] = rating.where(rating.between(1, 5))
    chunk['rating_missing'] = chunk['rating_clean'].isna()

    text_col = 'review_text' if 'review_text' in chunk.columns else 'review_text_incomplete'
    if text_col in chunk.columns:
        chunk['review_text_clean'] = normalize_review_text_series(chunk[text_col])
    else:
        chunk['review_text_clean'] = pd.NA
    chunk['has_review_text'] = chunk['review_text_clean'].notna()

    if 'is_read' in chunk.columns:
        chunk['is_read'] = chunk['is_read'].astype('boolean').fillna(False)
    else:
        chunk['is_read'] = False

    if {'started_at', 'read_at'}.issubset(chunk.columns):
        duration = (chunk['read_at'] - chunk['started_at']).dt.days
        chunk['reading_duration_days'] = duration.where(duration.between(0, 365))
    else:
        chunk['reading_duration_days'] = np.nan
    chunk['has_reading_duration'] = chunk['reading_duration_days'].notna()

    has_review = chunk['has_review_text']
    has_rating = chunk['rating_clean'].notna()
    is_read = chunk['is_read'].astype(bool)
    chunk['engagement_mode'] = np.select(
        [has_review, has_rating, is_read],
        ['review', 'rating_only', 'read_no_rating'],
        default='shelf_only',
    )
    return chunk


def summarize_interaction_chunk(chunk: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    rated = chunk[chunk['rating_clean'].notna()].copy()
    if len(rated):
        rated['rating_sq'] = np.square(rated['rating_clean'])
        user_part = rated.groupby('user_id').agg(
            rating_sum=('rating_clean', 'sum'),
            rating_sq_sum=('rating_sq', 'sum'),
            user_rating_count=('rating_clean', 'count'),
        ).reset_index()
    else:
        user_part = pd.DataFrame(columns=['user_id', 'rating_sum', 'rating_sq_sum', 'user_rating_count'])

    book_part = chunk.groupby('book_id').agg(
        interaction_count=('book_id', 'size'),
        explicit_rating_count=('rating_clean', 'count'),
        review_count=('has_review_text', 'sum'),
        shelf_only_count=('engagement_mode', lambda s: (s == 'shelf_only').sum()),
        read_count=('is_read', 'sum'),
        rating_sum=('rating_clean', 'sum'),
    ).reset_index()

    mode_counts = chunk['engagement_mode'].value_counts()
    return user_part, book_part, mode_counts

In [4]:
cleanup_output_files()

user_parts = []
book_parts = []
mode_counter = Counter()
total_interactions = 0
rating_sum_total = 0.0
rating_count_total = 0
seen_review_hashes = set()
duplicate_review_ids = 0

for chunk_idx, raw_chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTERACTION_CHUNKSIZE), start=1):
    chunk = process_interaction_chunk(raw_chunk)

    if 'review_id' in chunk.columns:
        review_ids = chunk['review_id'].dropna()
        review_hashes = pd.util.hash_pandas_object(review_ids, index=False).astype('uint64')
        duplicate_review_ids += int(review_hashes.duplicated().sum())
        chunk_hashes = set(review_hashes.tolist())
        duplicate_review_ids += len(chunk_hashes & seen_review_hashes)
        seen_review_hashes.update(chunk_hashes)

    user_part, book_part, mode_counts = summarize_interaction_chunk(chunk)
    user_parts.append(user_part)
    book_parts.append(book_part)
    mode_counter.update(mode_counts.to_dict())

    rating_sum_total += float(chunk['rating_clean'].sum(skipna=True))
    rating_count_total += int(chunk['rating_clean'].notna().sum())
    total_interactions += len(chunk)

    if chunk_idx % 10 == 0:
        print(f'First pass chunks={{chunk_idx:,}} interactions={{total_interactions:,}}')

    del raw_chunk, chunk, user_part, book_part
    gc.collect()

global_mean_rating = rating_sum_total / rating_count_total if rating_count_total else np.nan
print(f'First pass complete: {{total_interactions:,}} interactions')
print(f'Explicit ratings: {{rating_count_total:,}}; global mean={{global_mean_rating:.4f}}')
print(f'Duplicate review_id count observed: {{duplicate_review_ids:,}}')

/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

First pass chunks={chunk_idx:,} interactions={total_interactions:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

First pass chunks={chunk_idx:,} interactions={total_interactions:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

...

First pass chunks={chunk_idx:,} interactions={total_interactions:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

First pass chunks={chunk_idx:,} interactions={total_interactions:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

First pass complete: {total_interactions:,} interactions
Explicit ratings: {rating_count_total:,}; global mean={global_mean_rating:.4f}
Duplicate review_id count observed: {duplicate_review_ids:,}


## Section 3 — User and Book Interaction Features

Build user bias features and book-level interaction aggregates. Cold-start over all books will be finalized when each book chunk is merged with these aggregates.

In [5]:
user_partial = pd.concat(user_parts, ignore_index=True) if user_parts else pd.DataFrame(columns=['user_id', 'rating_sum', 'rating_sq_sum', 'user_rating_count'])
user_features = user_partial.groupby('user_id', as_index=False).agg(
    rating_sum=('rating_sum', 'sum'),
    rating_sq_sum=('rating_sq_sum', 'sum'),
    user_rating_count=('user_rating_count', 'sum'),
)
user_features['user_mean_rating'] = user_features['rating_sum'] / user_features['user_rating_count']
variance = (user_features['rating_sq_sum'] / user_features['user_rating_count']) - np.square(user_features['user_mean_rating'])
user_features['user_rating_std'] = np.sqrt(variance.clip(lower=0))
user_features['user_rating_bias'] = user_features['user_mean_rating'] - global_mean_rating
user_features = user_features[['user_id', 'user_mean_rating', 'user_rating_std', 'user_rating_count', 'user_rating_bias']]

book_partial = pd.concat(book_parts, ignore_index=True) if book_parts else pd.DataFrame(columns=['book_id'])
book_interaction_raw = book_partial.groupby('book_id', as_index=False).agg(
    interaction_count=('interaction_count', 'sum'),
    explicit_rating_count=('explicit_rating_count', 'sum'),
    review_count=('review_count', 'sum'),
    shelf_only_count=('shelf_only_count', 'sum'),
    read_count=('read_count', 'sum'),
    rating_sum=('rating_sum', 'sum'),
)
book_interaction_raw['mean_user_rating'] = book_interaction_raw['rating_sum'] / book_interaction_raw['explicit_rating_count'].replace(0, np.nan)
book_interaction_raw = book_interaction_raw.drop(columns=['rating_sum'])
book_interaction_raw['book_id'] = book_interaction_raw['book_id'].astype('string')

for col in ['interaction_count', 'explicit_rating_count', 'review_count', 'shelf_only_count', 'read_count']:
    book_interaction_raw[col] = book_interaction_raw[col].fillna(0).astype('int64')

print(f'user_features: {{user_features.shape}}')
print(f'book_interaction_raw: {{book_interaction_raw.shape}}')
display(user_features.head())
display(book_interaction_raw.head())

user_features: {user_features.shape}
book_interaction_raw: {book_interaction_raw.shape}


,user_id,user_mean_rating,user_rating_std,user_rating_count,user_rating_bias
0,00000377eea48021d3002730d56aca9a,4.400000,0.748331,25,0.480241
1,00004584d524ec468619e81b176cc991,4.105263,0.851917,38,0.185504
2,00009ab2ed8cbfceda5a59da40966321,4.000000,0.000000,3,0.080241
3,00009e46d18f223a82b22da38586b605,3.181818,0.715819,11,-0.737941
4,0000bad9195b66484e98f7f4be5f227a,3.000000,0.925820,7,-0.919759


,book_id,interaction_count,explicit_rating_count,review_count,shelf_only_count,read_count,mean_user_rating
0,10000034,2,2,1,0,2,2.500000
1,10000045,2,1,1,1,1,4.000000
2,10000600,1085,318,23,750,335,3.971698
3,10000781,9,6,1,3,6,3.666667
4,10000794,6,5,1,1,5,3.200000


## Section 4 — Interaction Second Pass: Curated Parquet

Write all interactions with cleaned ratings, engagement mode, reading duration, and user bias. Events without explicit ratings keep `rating_clean` as missing and are preserved as intent/behavior signals.

In [6]:
interaction_writer = None
second_pass_rows = 0
second_pass_mode_counter = Counter()

for chunk_idx, raw_chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTERACTION_CHUNKSIZE), start=1):
    chunk = process_interaction_chunk(raw_chunk)
    chunk = chunk.merge(user_features, on='user_id', how='left')
    chunk['user_rating_bias'] = chunk['user_rating_bias'].fillna(0)
    chunk['user_rating_count'] = chunk['user_rating_count'].fillna(0).astype('int64')

    second_pass_mode_counter.update(chunk['engagement_mode'].value_counts().to_dict())
    second_pass_rows += len(chunk)
    interaction_writer = write_parquet_chunk(INTERACTIONS_CURATED_PATH, chunk, interaction_writer)

    if chunk_idx % 10 == 0:
        print(f'Second pass chunks={{chunk_idx:,}} interactions={{second_pass_rows:,}}')

    del raw_chunk, chunk
    gc.collect()

if interaction_writer is not None:
    interaction_writer.close()

print(f'Wrote interactions: {{second_pass_rows:,}} rows -> {{INTERACTIONS_CURATED_PATH}}')

/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Second pass chunks={chunk_idx:,} interactions={second_pass_rows:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Second pass chunks={chunk_idx:,} interactions={second_pass_rows:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

...

Second pass chunks={chunk_idx:,} interactions={second_pass_rows:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Second pass chunks={chunk_idx:,} interactions={second_pass_rows:,}


/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_904175/3159480443.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Wrote interactions: {second_pass_rows:,} rows -> {INTERACTIONS_CURATED_PATH}


## Section 5 — Books Pipeline and Curated Parquet

Apply EDA-grounded cleaning to all book metadata in chunks, then merge book interaction aggregates so cold-start is calculated across the full book universe.

In [11]:
def process_books_chunk(raw_books: pd.DataFrame, seen_book_ids: set[str]) -> tuple[pd.DataFrame, int]:
    books = empty_strings_to_na(raw_books.copy())
    safe_to_numeric_inplace(books, BOOK_NUMERIC_COLUMNS)

    books['book_id'] = books['book_id'].astype('string')
    if 'work_id' in books.columns:
        books['work_id'] = books['work_id'].astype('string')
    duplicate_in_chunk = int(books['book_id'].duplicated().sum())
    books = books.drop_duplicates(subset=['book_id'], keep='first')
    already_seen = books['book_id'].isin(seen_book_ids)
    duplicate_across_chunks = int(already_seen.sum())
    books = books.loc[~already_seen].copy()
    seen_book_ids.update(books['book_id'].dropna().astype(str).tolist())

    if 'is_ebook' in books.columns:
        books['is_ebook'] = parse_bool_column(books['is_ebook'])

    for col in ['title', 'title_without_series', 'description', 'publisher', 'format', 'language_code', 'country_code', 'isbn', 'isbn13', 'asin', 'kindle_asin']:
        if col in books.columns:
            books[col] = clean_string_column(books[col])

    books['language_code_clean'] = books.get('language_code', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().replace('', pd.NA)
    books['format_clean'] = books.get('format', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().str.strip().replace('', pd.NA)
    books['publisher_clean'] = books.get('publisher', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().str.strip().replace('', pd.NA)

    for id_col in ['isbn', 'isbn13', 'asin', 'kindle_asin']:
        books[f'has_{id_col}'] = books[id_col].notna() if id_col in books.columns else False

    author_features = features_from_nested(books['authors'], extract_author_features) if 'authors' in books.columns else pd.DataFrame(index=books.index)
    series_features = features_from_nested(books['series'], extract_series_features) if 'series' in books.columns else pd.DataFrame(index=books.index)
    shelf_features = features_from_nested(books['popular_shelves'], extract_shelf_features) if 'popular_shelves' in books.columns else pd.DataFrame(index=books.index)
    similar_features = features_from_nested(books['similar_books'], extract_similar_books_features) if 'similar_books' in books.columns else pd.DataFrame(index=books.index)
    books = pd.concat([books, author_features, series_features, shelf_features, similar_features], axis=1)

    pub_year = pd.to_numeric(books.get('publication_year', pd.Series(np.nan, index=books.index)), errors='coerce')
    books['publication_year_clean'] = pub_year.where(pub_year.between(PUBLICATION_YEAR_MIN, PUBLICATION_YEAR_MAX))
    books['publication_period'] = pd.cut(
        books['publication_year_clean'],
        bins=PUBLICATION_PERIOD_BINS,
        labels=PUBLICATION_PERIOD_LABELS,
        right=False,
    ).astype('string')

    theme_cols = [f'theme_{theme}' for theme in THEME_GROUPS]
    for col in theme_cols:
        books[col] = books[col].fillna(False).astype(bool)
    books[THEME_COUNT_COL] = books[THEME_COUNT_COL].fillna(0).astype('int16')
    books['series_count'] = books['series_count'].fillna(0).astype('int16')
    books['is_in_series'] = books['is_in_series'].fillna(False).astype(bool)

    books = books.drop(columns=[col for col in ['authors', 'series', 'popular_shelves', 'similar_books'] if col in books.columns])
    books = add_log_and_cap_features(books, ['ratings_count', 'text_reviews_count', 'to_read_count', 'num_pages'])

    books = books.merge(book_interaction_raw, on='book_id', how='left')
    for col in ['interaction_count', 'explicit_rating_count', 'review_count', 'shelf_only_count', 'read_count']:
        books[col] = books[col].fillna(0).astype('int64')
    books['mean_user_rating'] = pd.to_numeric(books['mean_user_rating'], errors='coerce')
    books['is_cold_start_all_books'] = books['interaction_count'].lt(COLD_START_THRESHOLD)
    books['is_cold_start_interacted_books'] = books['interaction_count'].gt(0) & books['interaction_count'].lt(COLD_START_THRESHOLD)
    books['interaction_confidence_bucket'] = confidence_bucket(books['interaction_count'])
    books = add_log_and_cap_features(books, ['interaction_count'])
    return books, duplicate_in_chunk + duplicate_across_chunks

In [12]:
book_writer = None
book_rows = 0
book_duplicate_count = 0
seen_book_ids = set()
book_interaction_feature_parts = []
critical_null_counts = Counter()
cold_start_true_count = 0
cold_start_interacted_true_count = 0
interacted_book_count = 0

for chunk_idx, raw_books in enumerate(read_jsonl_chunks(cfg.books_file, chunksize=BOOK_CHUNKSIZE), start=1):
    books_chunk, duplicate_count = process_books_chunk(raw_books, seen_book_ids)
    book_duplicate_count += duplicate_count
    book_rows += len(books_chunk)
    cold_start_true_count += int(books_chunk['is_cold_start_all_books'].sum())
    interacted_mask = books_chunk['interaction_count'].gt(0)
    interacted_book_count += int(interacted_mask.sum())
    cold_start_interacted_true_count += int(books_chunk.loc[interacted_mask, 'is_cold_start_interacted_books'].sum())

    for col in ['book_id', 'primary_author_id_role_filtered', 'publication_year_clean', 'to_read_count']:
        critical_null_counts[f'books.{{col}}'] += int(books_chunk[col].isna().sum())

    book_interaction_feature_parts.append(books_chunk[[
        'book_id',
        'interaction_count',
        'explicit_rating_count',
        'review_count',
        'shelf_only_count',
        'read_count',
        'mean_user_rating',
        'is_cold_start_all_books',
        'is_cold_start_interacted_books',
        'interaction_confidence_bucket',
    ]].copy())

    book_writer = write_parquet_chunk(BOOKS_CURATED_PATH, books_chunk, book_writer)

    if chunk_idx % 5 == 0:
        print(f'Book chunks={{chunk_idx:,}} books={{book_rows:,}}')

    del raw_books, books_chunk
    gc.collect()

if book_writer is not None:
    book_writer.close()

book_interaction_features = pd.concat(book_interaction_feature_parts, ignore_index=True)
safe_write_parquet(book_interaction_features, BOOK_INTERACTION_FEATURES_PATH)
safe_write_parquet(user_features, USER_FEATURES_PATH)

print(f'Wrote books: {{book_rows:,}} rows -> {{BOOKS_CURATED_PATH}}')
print(f'Book duplicate count observed: {{book_duplicate_count:,}}')

Wrote books: {book_rows:,} rows -> {BOOKS_CURATED_PATH}
Book duplicate count observed: {book_duplicate_count:,}


## Section 6 — Summary

Write a JSON summary for reproducibility and quick downstream inspection.

In [13]:
engagement_mode_distribution = {key: int(value) for key, value in sorted(second_pass_mode_counter.items())}
curation_summary = {
    'category': CATEGORY,
    'display_name': cfg.display_name,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'books_rows': int(book_rows),
    'interactions_rows': int(second_pass_rows),
    'user_features_rows': int(len(user_features)),
    'book_interaction_features_rows': int(len(book_interaction_features)),
    'global_mean_rating': None if pd.isna(global_mean_rating) else float(global_mean_rating),
    'explicit_rating_count': int(rating_count_total),
    'engagement_mode_distribution': engagement_mode_distribution,
    'critical_null_counts': dict(critical_null_counts),
    'cold_start_all_books_pct': cold_start_true_count / book_rows if book_rows else 0.0,
    'cold_start_interacted_books_pct': cold_start_interacted_true_count / interacted_book_count if interacted_book_count else 0.0,
    'duplicate_review_ids_observed': int(duplicate_review_ids),
    'duplicate_book_ids_observed': int(book_duplicate_count),
    'output_files': {
        'books_curated': str(BOOKS_CURATED_PATH),
        'interactions_curated': str(INTERACTIONS_CURATED_PATH),
        'user_features': str(USER_FEATURES_PATH),
        'book_interaction_features': str(BOOK_INTERACTION_FEATURES_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(curation_summary, ensure_ascii=False, indent=2))
print(json.dumps(curation_summary, ensure_ascii=False, indent=2)[:2000])

{
  "category": "young_adult",
  "display_name": "Young Adult",
  "created_at_utc": "2026-04-26T13:31:07.032047+00:00",
  "books_rows": 93398,
  "interactions_rows": 34919254,
  "user_features_rows": 567806,
  "book_interaction_features_rows": 93398,
  "global_mean_rating": 3.919759341424071,
  "explicit_rating_count": 14731908,
  "engagement_mode_distribution": {
    "rating_only": 12414298,
    "read_no_rating": 947758,
    "review": 2400272,
    "shelf_only": 19156926
  },
  "critical_null_counts": {
    "books.{col}": 16494
  },
  "cold_start_all_books_pct": 0.2836356238891625,
  "cold_start_interacted_books_pct": 0.2836356238891625,
  "duplicate_review_ids_observed": 0,
  "duplicate_book_ids_observed": 0,
  "output_files": {
    "books_curated": "/home/nakato/projects/BigBook/data/processed/young_adult/books_curated.parquet",
    "interactions_curated": "/home/nakato/projects/BigBook/data/processed/young_adult/interactions_curated.parquet",
    "user_features": "/home/nakato/proje

## Section 7 — Validation

Assertions mirror the EDA-grounded test plan and protect the downstream recommender from silent data-contract drift. Large parquet files are validated by reading only the needed columns.

In [14]:
assert BOOKS_CURATED_PATH.exists()
assert INTERACTIONS_CURATED_PATH.exists()
assert USER_FEATURES_PATH.exists()
assert BOOK_INTERACTION_FEATURES_PATH.exists()
assert SUMMARY_PATH.exists()
assert duplicate_review_ids == 0, f'Duplicate review_id values observed: {duplicate_review_ids}'
assert book_duplicate_count == 0, f'Duplicate book_id values observed: {book_duplicate_count}'
assert user_features['user_rating_bias'].notna().all()
assert set(second_pass_mode_counter).issubset(VALID_ENGAGEMENT_MODES)

theme_cols = [f'theme_{theme}' for theme in THEME_GROUPS]
books_check_cols = ['book_id', 'publication_year_clean', THEME_COUNT_COL, 'interaction_count', 'is_cold_start_all_books', 'is_cold_start_interacted_books'] + theme_cols
books_check = pd.read_parquet(BOOKS_CURATED_PATH, columns=books_check_cols)
assert books_check['book_id'].notna().all()
assert books_check['book_id'].duplicated().sum() == 0
assert books_check['publication_year_clean'].dropna().between(PUBLICATION_YEAR_MIN, PUBLICATION_YEAR_MAX).all()
for col in theme_cols:
    assert books_check[col].dtype == bool
assert THEME_COUNT_COL in books_check.columns
assert books_check.loc[books_check['interaction_count'].eq(0), 'is_cold_start_all_books'].all()
assert books_check['is_cold_start_all_books'].dtype == bool
assert books_check['is_cold_start_interacted_books'].dtype == bool

interactions_check = pd.read_parquet(INTERACTIONS_CURATED_PATH, columns=['rating_clean', 'engagement_mode', 'reading_duration_days', 'user_rating_bias'])
assert interactions_check['rating_clean'].dropna().between(1, 5).all()
assert interactions_check['engagement_mode'].isin(VALID_ENGAGEMENT_MODES).all()
assert interactions_check['reading_duration_days'].dropna().between(0, 365).all()
assert interactions_check['user_rating_bias'].notna().all()

print('All validations passed!')

All validations passed!
